# "가격을 맞혀봐요!" 캡스톤 프로젝트

이번 주 목표 - Amazon 데이터 스크랩을 기반으로 상품 설명에서 가격을 예측하는 모델 만들기

# 진행 순서

1일차: 데이터 수집 및 정제  
2일차: 데이터 전처리  
3일차: 평가, 기준 모델, 전통적 ML  
4일차: 딥러닝과 LLM  
5일차: 프론티어 모델 파인튜닝  

## 5일차: 프론티어 모델 파인튜닝

이제 OpenAI API를 사용하여 GPT-4.1-nano의 나만의 파인튜닝 버전을 만들어 봅니다

In [ ]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [ ]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()

# 데이터 크기

OpenAI는 50~100개 예시로 파인튜닝하는 것을 권장합니다

저는 20,000개로 진행했습니다.

제 비용은 $3.42였습니다 - 여러분은 100개로 진행하면 비용이 매우 적게 듭니다!

In [ ]:
# OpenAI는 50~100개 예시로 파인튜닝을 권장합니다
# 예시가 매우 작으므로 100개로 진행합니다 (에폭 1회)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [ ]:
len(fine_tune_train)

# 1단계

파인튜닝용 데이터를 JSONL(JSON Lines) 형식으로 준비하고 OpenAI에 업로드합니다

In [ ]:
def messages_for(item):
    # 한국어 프롬프트로 파인튜닝 데이터를 구성합니다
    message = f"이 제품의 가격을 예측하세요. 가격만 답하고 설명은 하지 마세요.\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
messages_for(fine_tune_train[0])

In [ ]:
# 항목들을 JSON 객체 리스트(jsonl 문자열)로 변환합니다
# 각 행은 다음 형식의 메시지를 나타냅니다:
# {"messages" : [{"role": "system", "content": "가격을 예측하세요...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
print(make_jsonl(train[:3]))

In [ ]:
# 항목들을 jsonl로 변환하고 파일로 저장

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
train_file

In [ ]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
validation_file

https://platform.openai.com/storage/files/

# 2단계

## 이제 파인튜닝을 시작합니다!

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
job_id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

https://platform.openai.com/finetune


# 3단계

파인튜닝된 모델을 테스트합니다

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
# 프롬프트

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [ ]:
# 테스트해 봅니다

test_messages_for(test[0])

In [ ]:
# 추론 함수


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

In [ ]:
# 참고 결과 (평균 오차)
# 96.58 - mini 200개
# 79.29 - mini 2000개
# 82.26 - nano 2000개
# 67.75 - nano 20,000개